# TonyPi 路径规划与自主导航测试

本 Notebook 在 demo1～demo3 的基础上完成实验四：AprilTag + PnP 定位、二维栅格地图、A* 路径规划和闭环导航。它只负责让机器人从当前位置自主走到指定 `(x, y)` 附近，不包含比赛状态机或其他交互功能。

请在平整场地中按顺序运行单元。真实动作前务必确认机器人周围没有人员、电线或临时障碍物。

## 1. 导入库并设置路径

加载路径规划、图像处理和 TonyPi 硬件库。本 Notebook 的实验参数、相机标定和 Tag 世界坐标均在后续单元中直接定义。

In [ ]:
import heapq
import math
import sys
import time
from pathlib import Path

import apriltag
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

TONYPI_ROOT = Path('/home/pi/TonyPi')
for path in (TONYPI_ROOT, TONYPI_ROOT / 'HiwonderSDK'):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

import hiwonder.ActionGroupControl as AGC
import hiwonder.Camera as Camera
import hiwonder.ros_robot_controller_sdk as rrc
import hiwonder.yaml_handle as yaml_handle
from hiwonder.Controller import Controller

print('基础库与 TonyPi 硬件库加载完成')

## 2. 加载场地地图与参数

这里直接定义地图、导航、相机、动作、相机标定和 Tag 世界坐标，不在运行时读取外部配置或标定文件。学生现场通常只需修改最后两行目标坐标。

In [ ]:
# =========================
# 地图与安全参数
# =========================
FIELD_WIDTH_CM = 300.0
FIELD_HEIGHT_CM = 300.0
GRID_RESOLUTION_CM = 5.0
OBSTACLE_INFLATION_CM = 30.0
HARD_CLEARANCE_CM = 17.0
BOUNDARY_CLEARANCE_CM = 17.0
OBSTACLE_COST_MAX = 80.0
LINE_CLEAR_MAX_COST = 60.0

# =========================
# 闭环导航参数
# =========================
ARRIVAL_RADIUS_CM = 12.0
TURN_TOLERANCE_DEG = 20.0
WAYPOINT_LOOKAHEAD_CM = 30.0
MAX_NAV_STEPS = 80
MAX_LOCALIZE_RETRIES = 5
MAX_FORWARD_ACTIONS_PER_STEP = 3
MAX_TURN_ACTIONS_PER_STEP = 2
DISPLAY_EACH_NAV_STEP = True

# =========================
# 相机与头部参数
# =========================
HEAD_CENTER_ANGLE = 100.0
HEAD_LEFT_ANGLE = 145.0
HEAD_RIGHT_ANGLE = 55.0
HEAD_MOVE_MS = 600
HEAD_SETTLE_S = 0.55
CAMERA_DISCARD_FRAMES = 3
CAMERA_FRAME_GAP_S = 0.08
CAMERA_FORWARD_OFFSET_CM = 0.0

# =========================
# TonyPi 动作参数
# =========================
ACTION_GROUP_DIR = TONYPI_ROOT / 'ActionGroups'
STAND_GROUP = 'stand'
FORWARD_GROUP = 'go_forward_fast'
TURN_LEFT_GROUP = 'turn_left_small_step_s80'
TURN_RIGHT_GROUP = 'turn_right_small_step_s80'
FORWARD_ACTION_CM = 3.5
TURN_LEFT_ACTION_DEG = 7.5
TURN_RIGHT_ACTION_DEG = 7.5

# =========================
# 相机标定参数（与 demo3 一致）
# =========================
camera_matrix = np.array([
    [442.7764263403395, 0.0, 312.0124313067304],
    [0.0, 442.27334821002967, 221.54782360751392],
    [0.0, 0.0, 1.0],
], dtype=np.float64)
dist_coeffs = np.array(
    [-0.3563573203841699, 0.16913777232573327, 0.0, 0.0, 0.0],
    dtype=np.float64,
)

# =========================
# AprilTag 世界坐标（与 demo3 一致）
# =========================
def load_tag_pos():
    def expand(position):
        origin = np.append(position, np.array([0.0]))
        corners = np.array(
            [[0, 0, 0], [5, 0, 0], [5, 5, 0], [0, 5, 0]],
            dtype=np.float64,
        )
        return corners + origin

    x_negative = {'1', '7', '10', '13', '19', '21', '26', '29', '35'}
    y_negative = {'4', '6', '9', '14', '18', '22', '28', '32', '36'}
    x_positive = {'3', '5', '12', '16', '17', '23', '27', '31', '33'}
    y_positive = {'2', '8', '11', '15', '20', '24', '25', '30', '34'}
    x_neg_shift = np.array([[0, 0, 0], [0, -5, 0], [0, -5, -5], [0, 0, -5]])
    y_neg_shift = np.array([[0, 0, 0], [5, 0, 0], [5, 0, -5], [0, 0, -5]])
    x_pos_shift = np.array([[0, 0, 0], [0, 5, 0], [0, 5, -5], [0, 0, -5]])
    y_pos_shift = np.array([[0, 0, 0], [-5, 0, 0], [-5, 0, -5], [0, 0, -5]])

    def expand_wall(tag_id, position):
        origin = np.append(position, np.array([39.8]))
        if tag_id in x_negative:
            return x_neg_shift + origin
        if tag_id in y_negative:
            return y_neg_shift + origin
        if tag_id in x_positive:
            return x_pos_shift + origin
        if tag_id in y_positive:
            return y_pos_shift + origin
        raise ValueError('未知的墙面 Tag ID：{}'.format(tag_id))

    floor_positions = {
        '37': [166, 287], '38': [231.3, 287.5], '39': [9.5, 250],
        '40': [48.7, 266], '41': [279.5, 255], '42': [278, 182],
        '43': [271.5, 36], '44': [263, 108], '45': [225, 126],
        '46': [226.1, 62], '47': [114.2, 60], '48': [65.5, 23],
        '49': [22.5, 24.5], '50': [80, 95], '51': [12.4, 142],
        '52': [12.5, 187], '53': [126, 237], '54': [143, 141],
        '55': [80.5, 148], '56': [164, 54.4], '57': [239, 213.5],
        '58': [205, 241], '59': [199.5, 141.5], '60': [78.5, 236.5],
        '61': [122, 156.5],
    }
    wall_positions = {
        '1': ('1', [196, 30]), '2': ('4', [196, 5]),
        '3': ('3', [221, 5]), '4': ('2', [221, 30]),
        '5': ('7', [116.5, 31.5]), '6': ('6', [116.5, 6.5]),
        '7': ('5', [141.5, 6.5]), '8': ('8', [141.5, 31.5]),
        '9': ('10', [14.5, 79.5]), '10': ('9', [14.5, 54.5]),
        '11': ('12', [39.5, 54.5]), '12': ('11', [39.5, 79.5]),
        '13': ('13', [69.5, 206]), '14': ('14', [69.5, 181]),
        '15': ('16', [94.5, 181]), '16': ('15', [94.5, 206]),
        '17': ('19', [92, 288]), '18': ('18', [92, 263]),
        '19': ('17', [117, 263]), '20': ('20', [117, 288]),
        '21': ('21', [154.5, 209]), '22': ('22', [154.5, 184]),
        '23': ('23', [179.5, 184]), '24': ('24', [179.5, 209]),
        '25': ('26', [235.0, 271.5]), '26': ('28', [235.0, 246.5]),
        '27': ('27', [260.0, 246.5]), '28': ('25', [260.0, 271.5]),
        '29': ('29', [235.5, 170.8]), '30': ('32', [235.5, 145.8]),
        '31': ('31', [260.5, 145.8]), '32': ('30', [260.5, 170.8]),
        '33': ('35', [158, 110]), '34': ('36', [158, 85]),
        '35': ('33', [183, 85]), '36': ('34', [183, 110]),
    }

    tag_poses = {tag_id: expand(position) for tag_id, position in floor_positions.items()}
    tag_poses.update({
        tag_id: expand_wall(wall_id, position)
        for tag_id, (wall_id, position) in wall_positions.items()
    })
    return tag_poses


tag_world_points = load_tag_pos()

# =========================
# 学生通常只修改这里
# =========================
TARGET_X = 150.0
TARGET_Y = 220.0

print(f'场地：{FIELD_WIDTH_CM:.0f} cm × {FIELD_HEIGHT_CM:.0f} cm')
print(f'栅格：{GRID_RESOLUTION_CM:.1f} cm')
print(f'障碍物膨胀：{OBSTACLE_INFLATION_CM:.1f} cm')
print(f'已加载 {len(tag_world_points)} 个 Tag')
print(f'前进动作：{FORWARD_GROUP}，约 {FORWARD_ACTION_CM:.1f} cm/次')
print(f'转向动作：{TURN_LEFT_GROUP} / {TURN_RIGHT_GROUP}')

## 3. 创建 AprilTag 检测器并打开相机

初始化与 demo1～demo3 相同的相机、舵机控制和 `tag36h11` 检测器。相机只需打开一次。

In [ ]:
detector = apriltag.Detector(apriltag.DetectorOptions(families='tag36h11'))
board = rrc.Board()
ctl = Controller(board)
servo_data = yaml_handle.get_yaml_data(yaml_handle.servo_file_path)
camera = Camera.Camera()
camera.camera_open()
camera_opened = True
time.sleep(1.0)
print('AprilTag 检测器、头部舵机和相机已准备完成')

## 4. 定义机器人定位函数

PnP 数学与 demo3 保持一致：Notebook 内部的 Tag 世界角点与检测器角点一一对应，通过 `-Rᵀt` 得到位置，并由相机光轴得到 yaw。左右转头时会扣除头部角度，换算回机器人本体朝向。若没有 Tag，则按“正前方→左→右→小角度转动机器人”的有限流程重试。

In [ ]:
def normalize_angle_deg(angle):
    return (float(angle) + 180.0) % 360.0 - 180.0


def set_head_pan(angle):
    angle = max(HEAD_RIGHT_ANGLE, min(HEAD_LEFT_ANGLE, float(angle)))
    pulse = int(1500 + (angle - 100.0) * 10.0)
    pulse += int(servo_data.get('servo2', 1500)) - 1500
    pulse = max(500, min(2500, pulse))
    ctl.set_pwm_servo_pulse(2, pulse, HEAD_MOVE_MS)
    time.sleep(HEAD_MOVE_MS / 1000.0 + HEAD_SETTLE_S)


def capture_settled_frame():
    frame = None
    ret = False
    for _ in range(CAMERA_DISCARD_FRAMES + 1):
        ret, frame = camera.read()
        time.sleep(CAMERA_FRAME_GAP_S)
    return frame if ret and frame is not None else None


def collect_correspondences(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    detections = detector.detect(gray)
    object_points = []
    image_points = []
    used_ids = []
    for tag in detections:
        tag_id = str(tag.tag_id)
        if tag_id not in tag_world_points:
            continue
        object_points.extend(tag_world_points[tag_id][:4])
        image_points.extend(np.asarray(tag.corners, dtype=np.float64))
        used_ids.append(int(tag_id))
    return (np.asarray(object_points, dtype=np.float64),
            np.asarray(image_points, dtype=np.float64), used_ids)


def locate_frame_with_pnp(frame, head_pan_angle=HEAD_CENTER_ANGLE):
    object_points, image_points, used_ids = collect_correspondences(frame)
    if len(object_points) < 4:
        return None, used_ids
    success, rvec, tvec = cv2.solvePnP(
        object_points, image_points, camera_matrix, dist_coeffs
    )
    if not success:
        return None, used_ids
    rotation_matrix, _ = cv2.Rodrigues(rvec)
    camera_position = -rotation_matrix.T @ tvec
    camera_forward = rotation_matrix.T @ np.array([[0.0], [0.0], [1.0]])
    camera_yaw = math.degrees(math.atan2(camera_forward[1, 0],
                                         camera_forward[0, 0]))
    robot_yaw = normalize_angle_deg(
        camera_yaw - (float(head_pan_angle) - HEAD_CENTER_ANGLE)
    )
    robot_x = float(camera_position[0, 0]) - CAMERA_FORWARD_OFFSET_CM * math.cos(math.radians(robot_yaw))
    robot_y = float(camera_position[1, 0]) - CAMERA_FORWARD_OFFSET_CM * math.sin(math.radians(robot_yaw))
    pose = {'x': robot_x, 'y': robot_y, 'yaw': robot_yaw}
    return pose, used_ids


def safe_stand(message=None):
    print(message or f'执行动作：{STAND_GROUP} × 1')
    try:
        AGC.stopActionGroup()
    except Exception:
        pass
    try:
        AGC.runActionGroup(STAND_GROUP, times=1, with_stand=False)
    except Exception as exc:
        print(f'站立动作执行失败：{exc}')


def localize_with_recovery(max_retries=MAX_LOCALIZE_RETRIES):
    scan_angles = [HEAD_CENTER_ANGLE, HEAD_LEFT_ANGLE, HEAD_RIGHT_ANGLE]
    try:
        for attempt in range(max(1, int(max_retries))):
            for pan in scan_angles:
                print(f'定位尝试 {attempt + 1}/{max_retries}：头部角度 {pan:.0f}°')
                set_head_pan(pan)
                frame = capture_settled_frame()
                if frame is None:
                    continue
                pose, used_ids = locate_frame_with_pnp(frame, pan)
                if pose is not None:
                    print('定位成功：x={x:.1f}, y={y:.1f}, yaw={yaw:.1f}°, Tag={ids}'.format(
                        ids=used_ids, **pose))
                    return pose
            if attempt < max_retries - 1:
                group = TURN_LEFT_GROUP if attempt % 2 == 0 else TURN_RIGHT_GROUP
                print(f'仍未找到可定位 Tag，执行小角度搜索动作：{group} × 1')
                set_head_pan(HEAD_CENTER_ANGLE)
                AGC.runActionGroup(group, times=1, with_stand=False)
                safe_stand()
        print('定位失败：已达到最大尝试次数，机器人停止。')
        safe_stand()
        return None
    finally:
        try:
            set_head_pan(HEAD_CENTER_ANGLE)
        except Exception:
            pass

## 5. 建立二维栅格地图

1～36 号 Tag 每四个属于同一个 `25 cm × 25 cm` 白色障碍物。本单元从 Notebook 内部的 Tag 四角坐标推导 9 个障碍物边界，再加入上方直接定义的膨胀距离。数组固定使用 `grid[x_index, y_index]`。

In [ ]:
GRID_NX = int(math.ceil(FIELD_WIDTH_CM / GRID_RESOLUTION_CM))
GRID_NY = int(math.ceil(FIELD_HEIGHT_CM / GRID_RESOLUTION_CM))


def derive_obstacle_bounds(tag_positions):
    grouped = {}
    for tag_id_text, corners_3d in tag_positions.items():
        tag_id = int(tag_id_text)
        if not 1 <= tag_id <= 36:
            continue
        group_id = (tag_id - 1) // 4
        xy = np.asarray(corners_3d, dtype=np.float64)[:4, :2]
        item = grouped.setdefault(group_id, {
            'x_min': float('inf'), 'x_max': -float('inf'),
            'y_min': float('inf'), 'y_max': -float('inf'),
        })
        item['x_min'] = min(item['x_min'], float(np.min(xy[:, 0])))
        item['x_max'] = max(item['x_max'], float(np.max(xy[:, 0])))
        item['y_min'] = min(item['y_min'], float(np.min(xy[:, 1])))
        item['y_max'] = max(item['y_max'], float(np.max(xy[:, 1])))
    if len(grouped) != 9:
        raise ValueError(f'应推导出 9 个障碍物，实际得到 {len(grouped)} 个')
    return grouped


def distance_to_rectangle(x, y, bounds):
    dx = max(bounds['x_min'] - x, 0.0, x - bounds['x_max'])
    dy = max(bounds['y_min'] - y, 0.0, y - bounds['y_max'])
    return math.hypot(dx, dy)


def obstacle_cost_for_distance(distance_cm):
    if distance_cm > OBSTACLE_INFLATION_CM:
        return 0.0
    if distance_cm <= HARD_CLEARANCE_CM:
        return OBSTACLE_COST_MAX
    soft_span = max(1e-6, OBSTACLE_INFLATION_CM - HARD_CLEARANCE_CM)
    ratio = 1.0 - (distance_cm - HARD_CLEARANCE_CM) / soft_span
    return OBSTACLE_COST_MAX * max(0.0, ratio) ** 2


def build_occupancy_grids():
    obstacle = np.zeros((GRID_NX, GRID_NY), dtype=bool)
    hard_blocked = np.zeros_like(obstacle)
    inflated = np.zeros_like(obstacle)
    cost = np.zeros((GRID_NX, GRID_NY), dtype=np.float64)
    for gx in range(GRID_NX):
        for gy in range(GRID_NY):
            x = (gx + 0.5) * GRID_RESOLUTION_CM
            y = (gy + 0.5) * GRID_RESOLUTION_CM
            if (x < BOUNDARY_CLEARANCE_CM or x > FIELD_WIDTH_CM - BOUNDARY_CLEARANCE_CM or
                    y < BOUNDARY_CLEARANCE_CM or y > FIELD_HEIGHT_CM - BOUNDARY_CLEARANCE_CM):
                hard_blocked[gx, gy] = True
                inflated[gx, gy] = True
                cost[gx, gy] = OBSTACLE_COST_MAX
            for bounds in OBSTACLE_BOUNDS.values():
                inside = (bounds['x_min'] <= x <= bounds['x_max'] and
                          bounds['y_min'] <= y <= bounds['y_max'])
                if inside:
                    obstacle[gx, gy] = True
                distance = distance_to_rectangle(x, y, bounds)
                cost[gx, gy] = max(cost[gx, gy], obstacle_cost_for_distance(distance))
                if distance <= HARD_CLEARANCE_CM:
                    hard_blocked[gx, gy] = True
                if distance <= OBSTACLE_INFLATION_CM:
                    inflated[gx, gy] = True
    hard_blocked |= obstacle
    inflated |= hard_blocked
    return obstacle, hard_blocked, inflated, cost


OBSTACLE_BOUNDS = derive_obstacle_bounds(tag_world_points)
OBSTACLE_GRID, HARD_BLOCKED_GRID, INFLATED_GRID, OBSTACLE_COST_GRID = build_occupancy_grids()
print(f'已建立 {GRID_NX} × {GRID_NY} 栅格，并从墙面 Tag 推导出 {len(OBSTACLE_BOUNDS)} 个障碍物')

## 6. 显示地图

灰色为障碍物本体，红色为硬安全区，橙色为软膨胀代价区。由于数组第一维表示 x、第二维表示 y，传给 matplotlib 时必须使用 `.T`；同时设置 `origin='lower'`，才能让图中的 x/y 与 PnP 世界坐标一致。

In [ ]:
def show_map(current_pose=None, target_xy=None, path=None, title='实验四栅格地图'):
    fig, ax = plt.subplots(figsize=(7, 7))
    inflation_only = INFLATED_GRID & ~HARD_BLOCKED_GRID
    hard_only = HARD_BLOCKED_GRID & ~OBSTACLE_GRID
    inflation_mask = np.ma.masked_where(~inflation_only.T, inflation_only.T)
    hard_mask = np.ma.masked_where(~hard_only.T, hard_only.T)
    obstacle_mask = np.ma.masked_where(~OBSTACLE_GRID.T, OBSTACLE_GRID.T)
    extent = [0, FIELD_WIDTH_CM, 0, FIELD_HEIGHT_CM]
    ax.imshow(inflation_mask, origin='lower', extent=extent, interpolation='nearest',
              cmap=ListedColormap(['#f4a261']), alpha=0.45)
    ax.imshow(hard_mask, origin='lower', extent=extent, interpolation='nearest',
              cmap=ListedColormap(['#e76f51']), alpha=0.65)
    ax.imshow(obstacle_mask, origin='lower', extent=extent, interpolation='nearest',
              cmap=ListedColormap(['#555555']), alpha=0.95)
    if path:
        xs, ys = zip(*path)
        ax.plot(xs, ys, 'b.-', linewidth=2, markersize=4, label='A* 路径')
    if current_pose is not None:
        x, y, yaw = current_pose['x'], current_pose['y'], current_pose['yaw']
        ax.plot(x, y, 'go', markersize=9, label='机器人')
        ax.arrow(x, y, 12 * math.cos(math.radians(yaw)),
                 12 * math.sin(math.radians(yaw)),
                 width=1.2, color='green', length_includes_head=True)
    if target_xy is not None:
        ax.plot(target_xy[0], target_xy[1], 'r*', markersize=15, label='目标点')
    ax.set_xlim(0, FIELD_WIDTH_CM)
    ax.set_ylim(0, FIELD_HEIGHT_CM)
    ax.set_aspect('equal')
    ax.set_xlabel('x / cm')
    ax.set_ylabel('y / cm')
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    plt.show()


show_map(target_xy=(TARGET_X, TARGET_Y))

## 7. 定义 A* 路径规划

A* 使用 8 邻域，直线代价为 `1`、对角线代价为 `sqrt(2)`，启发函数为欧氏距离。对角移动还会检查两侧直线格，避免从两个障碍格的角缝中穿过。返回值使用世界坐标。

In [ ]:
def in_field(xy):
    return 0.0 <= float(xy[0]) < FIELD_WIDTH_CM and 0.0 <= float(xy[1]) < FIELD_HEIGHT_CM


def world_to_grid(xy):
    return (int(float(xy[0]) // GRID_RESOLUTION_CM),
            int(float(xy[1]) // GRID_RESOLUTION_CM))


def grid_to_world(node):
    return ((node[0] + 0.5) * GRID_RESOLUTION_CM,
            (node[1] + 0.5) * GRID_RESOLUTION_CM)


def grid_is_free(node):
    gx, gy = node
    return (0 <= gx < GRID_NX and 0 <= gy < GRID_NY and
            not HARD_BLOCKED_GRID[gx, gy] and
            OBSTACLE_COST_GRID[gx, gy] < LINE_CLEAR_MAX_COST)


def validate_goal(goal_xy):
    if not in_field(goal_xy):
        raise ValueError(f'目标点 {goal_xy} 超出场地范围')
    node = world_to_grid(goal_xy)
    if OBSTACLE_GRID[node]:
        raise ValueError(f'目标点 {goal_xy} 位于障碍物内部')
    if HARD_BLOCKED_GRID[node]:
        raise ValueError(f'目标点 {goal_xy} 位于机器人硬安全区或边界禁止通行区域')
    if OBSTACLE_COST_GRID[node] >= LINE_CLEAR_MAX_COST:
        raise ValueError(f'目标点 {goal_xy} 过于靠近障碍物')
    return True


def astar(start_xy, goal_xy):
    validate_goal(goal_xy)
    if not in_field(start_xy):
        raise ValueError(f'起点 {start_xy} 超出场地范围')
    start = world_to_grid(start_xy)
    goal = world_to_grid(goal_xy)
    if not grid_is_free(start):
        raise ValueError(f'当前定位点 {start_xy} 位于禁止通行区域，停止规划')
    if start == goal:
        return [(float(start_xy[0]), float(start_xy[1])),
                (float(goal_xy[0]), float(goal_xy[1]))]

    moves = [
        (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
        (-1, -1, math.sqrt(2)), (-1, 1, math.sqrt(2)),
        (1, -1, math.sqrt(2)), (1, 1, math.sqrt(2)),
    ]
    open_heap = [(math.hypot(goal[0] - start[0], goal[1] - start[1]), 0.0, start)]
    came_from = {}
    g_score = {start: 0.0}
    closed = set()

    while open_heap:
        _, current_g, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        if current == goal:
            nodes = [current]
            while current in came_from:
                current = came_from[current]
                nodes.append(current)
            nodes.reverse()
            path = [grid_to_world(node) for node in nodes]
            path[0] = (float(start_xy[0]), float(start_xy[1]))
            path[-1] = (float(goal_xy[0]), float(goal_xy[1]))
            return path
        closed.add(current)

        for dx, dy, step_cost in moves:
            nxt = (current[0] + dx, current[1] + dy)
            if not grid_is_free(nxt):
                continue
            if dx != 0 and dy != 0:
                if not grid_is_free((current[0] + dx, current[1])):
                    continue
                if not grid_is_free((current[0], current[1] + dy)):
                    continue
            tentative = current_g + step_cost + float(OBSTACLE_COST_GRID[nxt])
            if tentative >= g_score.get(nxt, float('inf')):
                continue
            came_from[nxt] = current
            g_score[nxt] = tentative
            heuristic = math.hypot(goal[0] - nxt[0], goal[1] - nxt[1])
            heapq.heappush(open_heap, (tentative + heuristic, tentative, nxt))
    return []

## 8. 测试单次路径规划与简化

直线可见的连续路径点会被合并；导航时再从安全直线上选取一定距离内的 waypoint，因此机器人不需要每走 `5 cm` 就停一次。此处只做软件规划，不执行真实动作。

In [ ]:
def line_is_clear(start_xy, end_xy, sample_step_cm=None):
    if not in_field(start_xy) or not in_field(end_xy):
        return False
    distance = math.hypot(end_xy[0] - start_xy[0], end_xy[1] - start_xy[1])
    step = sample_step_cm or GRID_RESOLUTION_CM / 2.0
    samples = max(1, int(math.ceil(distance / step)))
    for index in range(samples + 1):
        t = index / float(samples)
        point = (start_xy[0] + (end_xy[0] - start_xy[0]) * t,
                 start_xy[1] + (end_xy[1] - start_xy[1]) * t)
        if not in_field(point) or not grid_is_free(world_to_grid(point)):
            return False
    return True


def simplify_path(path):
    if len(path) <= 2:
        return list(path)
    simplified = [path[0]]
    index = 0
    while index < len(path) - 1:
        candidate = len(path) - 1
        while candidate > index + 1 and not line_is_clear(path[index], path[candidate]):
            candidate -= 1
        simplified.append(path[candidate])
        index = candidate
    return simplified


def choose_waypoint(current_xy, path, lookahead_cm=WAYPOINT_LOOKAHEAD_CM):
    candidates = [point for point in path[1:]
                  if math.hypot(point[0] - current_xy[0], point[1] - current_xy[1]) > 1.0]
    if not candidates:
        return path[-1]
    selected = candidates[0]
    for point in candidates:
        distance = math.hypot(point[0] - current_xy[0], point[1] - current_xy[1])
        if distance > lookahead_cm:
            break
        if line_is_clear(current_xy, point):
            selected = point
    return selected


TEST_START = (280.0, 40.0)
try:
    test_path = astar(TEST_START, (TARGET_X, TARGET_Y))
    test_path = simplify_path(test_path) if test_path else []
    print(f'单次规划完成：简化后 {len(test_path)} 个路径点')
    show_map({'x': TEST_START[0], 'y': TEST_START[1], 'yaw': 0.0},
             (TARGET_X, TARGET_Y), test_path, 'A* 单次规划测试')
except ValueError as exc:
    test_path = []
    print(f'单次规划测试未执行：{exc}')

## 9. 定义机器人动作函数

动作名、动作组目录和单次位移均在参数单元中直接定义。每轮最多执行少量有限动作，随后主循环会重新定位、重新规划。执行前会打印动作，路径检查失败时不会向前盲走。

In [ ]:
def ensure_action_groups_exist():
    action_dir = ACTION_GROUP_DIR
    required = {STAND_GROUP, FORWARD_GROUP, TURN_LEFT_GROUP, TURN_RIGHT_GROUP}
    missing = [name for name in sorted(required) if not (action_dir / f'{name}.d6a').exists()]
    if missing:
        raise FileNotFoundError(f'缺少 ActionGroup：{missing}；目录：{action_dir}')


def execute_turn(yaw_error_deg):
    if abs(yaw_error_deg) <= TURN_TOLERANCE_DEG:
        return True
    if yaw_error_deg > 0.0:
        group = TURN_LEFT_GROUP
        action_deg = TURN_LEFT_ACTION_DEG
    else:
        group = TURN_RIGHT_GROUP
        action_deg = TURN_RIGHT_ACTION_DEG
    excess = max(0.0, abs(yaw_error_deg) - TURN_TOLERANCE_DEG)
    times = max(1, min(MAX_TURN_ACTIONS_PER_STEP,
                       int(math.ceil(excess / max(1.0, action_deg)))))
    print(f'执行转向：{group} × {times}（角度误差 {yaw_error_deg:.1f}°）')
    AGC.runActionGroup(group, times=times, with_stand=False)
    safe_stand()
    return True


def execute_forward(pose, waypoint, distance_to_goal):
    waypoint_distance = math.hypot(waypoint[0] - pose['x'], waypoint[1] - pose['y'])
    usable_distance = min(waypoint_distance,
                          max(FORWARD_ACTION_CM, distance_to_goal - ARRIVAL_RADIUS_CM))
    times = max(1, min(MAX_FORWARD_ACTIONS_PER_STEP,
                       int(usable_distance / max(0.1, FORWARD_ACTION_CM))))
    planned_distance = times * FORWARD_ACTION_CM
    yaw_rad = math.radians(pose['yaw'])
    predicted_end = (pose['x'] + planned_distance * math.cos(yaw_rad),
                     pose['y'] + planned_distance * math.sin(yaw_rad))
    if not line_is_clear((pose['x'], pose['y']), predicted_end):
        print('前进路径安全检查失败，本轮不执行前进。')
        return False
    print(f'执行前进：{FORWARD_GROUP} × {times}（预计 {planned_distance:.1f} cm）')
    AGC.runActionGroup(FORWARD_GROUP, times=times, with_stand=True)
    return True

## 10. 定义闭环导航函数

每轮都执行“定位→判断距离→A*→选择 waypoint→少量动作”。动作后回到下一轮重新定位和重新规划。到达只检查 `(x, y)` 距离，不执行最终 yaw 对齐；定位、规划、硬件异常、Ctrl+C 和步数超限都会站立退出。

In [ ]:
def navigate_to_target(target_x, target_y):
    goal = (float(target_x), float(target_y))
    try:
        ensure_action_groups_exist()
        validate_goal(goal)
        print(f'开始闭环导航，目标位置：({goal[0]:.1f}, {goal[1]:.1f})')

        for step in range(MAX_NAV_STEPS):
            print(f'\n===== 导航步骤 {step + 1}/{MAX_NAV_STEPS} =====')
            pose = localize_with_recovery(MAX_LOCALIZE_RETRIES)
            if pose is None:
                print('导航失败：在有限次数内无法恢复定位。')
                return False

            distance_to_goal = math.hypot(goal[0] - pose['x'], goal[1] - pose['y'])
            print(f'距离目标：{distance_to_goal:.1f} cm')
            if distance_to_goal <= ARRIVAL_RADIUS_CM:
                safe_stand()
                print('已到达目标位置')
                return True

            path = astar((pose['x'], pose['y']), goal)
            if not path:
                print('导航失败：A* 未找到不穿越障碍物的路径。')
                return False
            simplified_path = simplify_path(path)
            waypoint = choose_waypoint((pose['x'], pose['y']), path)
            if not line_is_clear((pose['x'], pose['y']), waypoint):
                print('导航失败：当前点到 waypoint 的直线穿过禁止区域。')
                return False
            print(f'本轮 A* 路径点：{len(path)}，简化后：{len(simplified_path)}，waypoint={waypoint}')
            if DISPLAY_EACH_NAV_STEP:
                show_map(pose, goal, simplified_path, f'闭环导航步骤 {step + 1}')

            target_yaw = math.degrees(math.atan2(waypoint[1] - pose['y'],
                                                  waypoint[0] - pose['x']))
            yaw_error = normalize_angle_deg(target_yaw - pose['yaw'])
            if abs(yaw_error) > TURN_TOLERANCE_DEG:
                execute_turn(yaw_error)
            else:
                if not execute_forward(pose, waypoint, distance_to_goal):
                    print('导航失败：安全检查不允许本轮前进。')
                    return False

        print(f'导航失败：已达到最大导航步数 {MAX_NAV_STEPS}。')
        return False
    except KeyboardInterrupt:
        print('检测到 Ctrl+C，导航已停止。')
        return False
    except Exception as exc:
        print(f'导航异常：{type(exc).__name__}: {exc}')
        return False
    finally:
        safe_stand()

## 11. 设置目标点

目标坐标已集中放在第 2 节的 `TARGET_X`、`TARGET_Y`。本单元只做安全检查；若目标位于场外、障碍物或膨胀禁行区，会明确报错。

In [ ]:
try:
    validate_goal((TARGET_X, TARGET_Y))
    print(f'目标点有效：({TARGET_X:.1f}, {TARGET_Y:.1f})')
except ValueError as exc:
    print(f'目标点无效，请修改第 2 节的 TARGET_X / TARGET_Y：{exc}')

## 12. 开始自主导航

确认地图、目标点和机器人周围安全后运行。函数内部有最大步数和最大定位重试次数；每批动作后都会进入下一轮重新定位与 A* 规划。

In [ ]:
navigation_ok = navigate_to_target(TARGET_X, TARGET_Y)
print('导航结果：', '成功' if navigation_ok else '失败')

## 13. 关闭相机与清理

完成实验或中止后运行本单元，让机器人站立、头部回正并释放相机。

In [ ]:
safe_stand()
try:
    set_head_pan(HEAD_CENTER_ANGLE)
finally:
    if globals().get('camera_opened', False):
        camera.camera_close()
        camera_opened = False
print('机器人已停止，相机已关闭')